# Kaggle runner

Runs the project's own code (cloned from GitHub) on a Kaggle GPU. This notebook contains no logic of its
own: it prepares the data, then calls `src.benchmark`, or `src.train` followed by `src.evaluate`.

**Kaggle settings (right panel):** Accelerator = GPU (T4 or P100), Internet = on.

**How to run:** set the parameters below, then *Save Version → Save & Run All (Commit)*. The notebook runs in
the background (up to 12 hours) and everything written to `/kaggle/working` is kept in the version's
**Output** tab, ready to download.

The dataset is rebuilt in `/tmp` at every session from a pinned Hugging Face copy and checked against the
WILDS paper (`src/prepare_data.py`). `/tmp` is not saved: only results are.

In [ ]:
MODE = "benchmark"                 # "benchmark" or "train"
CONFIG = "configs/cnn_none.yaml"
SEED = 0
GIT_REF = "main"                   # branch, tag or commit of the code to run

# To continue an interrupted training run: attach the output of the interrupted
# version as an input of this notebook and give the path of its run folder, e.g.
# "/kaggle/input/<notebook-name>/artifacts/runs/cnn_none/seed0".
RESUME_FROM = None

REPO_URL = "https://github.com/jasonmarolany971-afk/camelyon17-hospital-shift.git"
DATA_ROOT = "/tmp/datasets"        # not saved, rebuilt at every session
WORK = "/kaggle/working"           # saved in the version's Output tab

In [ ]:
import os
import shutil

# Progress bars refresh at most once a minute: keeps the saved log small.
os.environ["TQDM_MININTERVAL"] = "60"


def sh(command: str) -> None:
    """Run a shell command, show its output, and stop the notebook if it fails."""
    print(f"$ {command}", flush=True)
    get_ipython().system(command)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"Command failed: {command}")


sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
sh(f"df -h {WORK} /tmp")
sh("nproc && free -g | head -2")

## Code and dependencies

Kaggle images already provide PyTorch, NumPy, pandas, scikit-learn and matplotlib. Only `wilds` (and the
download tools, if missing) are installed. Warnings from pip's dependency resolver about unrelated
preinstalled packages can be ignored as long as the next cell passes.

In [ ]:
os.chdir(WORK)
shutil.rmtree("repo", ignore_errors=True)
sh(f"git clone --quiet {REPO_URL} repo")
os.chdir(f"{WORK}/repo")
sh(f"git checkout --quiet {GIT_REF} && git log --oneline -1")
sh("pip install --quiet wilds==2.0.0 huggingface_hub pyarrow")
sh(f"python -m src.check_env --data-root {DATA_ROOT} || true")  # informative: shows GPU and free disk

## Data

Downloads the pinned copy file by file, converts each one to the official WILDS layout and deletes it
(`--delete-parquet`), then verifies the result. Expect `All checks passed.` at the end.

In [ ]:
sh(f"python -m src.prepare_data --data-root {DATA_ROOT} --delete-parquet "
   f"--provenance {WORK}/data_provenance.json")

## Run

- `benchmark`: measures data loading, computation (fp32 and mixed precision) and end-to-end speed, and
  estimates the time of a full run. Result in `/kaggle/working/benchmarks/`.
- `train`: trains with the given seed (selection on `val`), then evaluates the selected checkpoint once on
  `id_val`, `val` and `test`. Results in `/kaggle/working/artifacts/` (checkpoints, predictions) and
  `/kaggle/working/repo/results/runs/` (metrics, history).

In [ ]:
import yaml

name = yaml.safe_load(open(CONFIG))["name"]
runs_dir = f"{WORK}/artifacts/runs"
run_dir = f"{runs_dir}/{name}/seed{SEED}"

if MODE == "benchmark":
    sh(f"python -m src.benchmark --config {CONFIG} --data-root {DATA_ROOT} --out-dir {WORK}/benchmarks")
elif MODE == "train":
    if RESUME_FROM:
        shutil.copytree(RESUME_FROM, run_dir, dirs_exist_ok=True)
    resume = " --resume" if RESUME_FROM else ""
    sh(f"python -m src.train --config {CONFIG} --seed {SEED} --data-root {DATA_ROOT} --output-dir {runs_dir}{resume}")
    sh(f"python -m src.evaluate --run-dir {run_dir} --data-root {DATA_ROOT}")
else:
    raise ValueError(f"Unknown MODE {MODE!r}")

## What to bring back to the Mac

From the version's **Output** tab:
- `benchmarks/*.json` → `results/benchmarks/` in the repository;
- `repo/results/runs/<name>/seed<seed>/` (`metrics.json`, `history.csv`) → `results/runs/` in the repository;
- `artifacts/runs/<name>/seed<seed>/` (checkpoints and predictions, not versioned) → `artifacts/runs/` on the Mac.